# Brain Cancer Type Classification from Tumour Gene Activity Using Machine Learning

**Group members**

| No. | Name | Roll No. |
|---|---|---|
| 1 | Raghav Yadav | 240598 |
| 2 | Saksham Lodha | 240626 |
| 3 | Rayna Manchanda | 240601 |
| 4 | Darshita Yadav | 240981 |
| 5 | Prerit Shrivastava | 240593 |

## Problem Statement
Predict a glioma's molecular type (**IDHwt**, **IDHmut-non-codel** or **IDHmut-codel**) from tumour gene expression alone.

Doctors currently need separate lab tests for the IDH mutation and the 1p/19q codeletion to decide the type. RNA sequencing measures about 20,000 genes in one go, and both changes leave a trace in that data. If a model can read the trace, one measurement could stand in for several tests.

**Notebook flow**

| Step | What happens |
|---|---|
| 1 | Understanding the dataset |
| 2 | Exploratory data analysis (EDA) |
| 3 | Data preprocessing |
| 4 | Feature selection |
| 5 | Dataset splitting |
| 6 | Selection of ML algorithms |
| 7 | Model implementation and training |
| 8 | Model evaluation |
| 9 | Model comparison |
| 10 | Feature importance and result interpretation |

## Setup
Libraries, plot style and the fixed settings used everywhere. `SEED = 42` makes every split and model reproducible.

In [ ]:
# ---- Settings -------------------------------------------------------------
import os
os.environ["OMP_NUM_THREADS"] = "1"          # keeps the parallel grid search stable

import warnings
warnings.filterwarnings("ignore")            # hide library warnings so the output stays readable

# ---- Libraries ------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import f_classif

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score, matthews_corrcoef,
                             cohen_kappa_score, log_loss, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report)

# ---- Plot style (same clean look as the report) ----------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "axes.edgecolor": "#333333",
    "xtick.direction": "in",
    "ytick.direction": "in",
    "figure.dpi": 110,
})
BLUE, GREY, ORANGE = "#3f6fa8", "#9a9a9a", "#d9822b"
CLASS_COLOURS = {"IDHwt": BLUE, "IDHmut-non-codel": ORANGE, "IDHmut-codel": GREY}

# ---- Project constants -----------------------------------------------------
SEED = 42                                                    # same random split every run
RAW = "data/raw/"                                            # folder with the two source files
classes = ["IDHwt", "IDHmut-non-codel", "IDHmut-codel"]      # class order used everywhere (0, 1, 2)

## 1. Understanding the Dataset
Two public files from the TCGA glioma study (Ceccarelli et al., 2016):
- **Gene expression matrix** - the inputs. RNA-seq values (RSEM-normalised) for each patient and gene.
- **Table S1** - the patient table. The column `IDH/codel subtype` is the target.

The two files are joined on the TCGA patient ID. Nothing else is changed in the raw files.

### 1.1 Load the two files

In [ ]:
# Gene expression: one row per patient, one column per gene
expr = pd.read_csv(RAW + "LGG-GBM.gene_expression.normalized.txt", sep="\t", index_col=0)

# Patient table (Table S1): one row per patient, with the subtype label
meta = pd.read_excel(RAW + "TableS1.PatientData.20151020.v3.xlsx", sheet_name=0, header=1)
meta = meta.set_index("Case")

print("Expression matrix (patients x genes):", expr.shape)
print("Patients in Table S1:", len(meta))

### 1.2 Join inputs with the target

In [ ]:
labels = meta["IDH/codel subtype"].dropna()          # the target, 3 classes
ids = expr.index.intersection(labels.index)          # patients that have both expression and a label

X = expr.loc[ids]                                    # input features
y = labels.loc[ids]                                  # target variable
grade = meta.loc[ids, "Grade"]                       # tumour grade, used only to describe the data

print("Labelled patients:", len(labels))
print("Patients with expression and label:", X.shape[0])

### 1.3 Records, attributes and their types

In [ ]:
numeric_inputs = X.select_dtypes(include="number").shape[1]

print("Records (patients):          ", X.shape[0])
print("Input attributes (genes):    ", X.shape[1])
print("Numerical input attributes:  ", numeric_inputs)
print("Categorical input attributes:", X.shape[1] - numeric_inputs)
print()
print("Target variable: IDH/codel subtype")
print("Target classes:  ", sorted(y.unique()))

**What this tells us**
- **Inputs:** every input is a **numerical**, continuous gene-activity value. There are no categorical inputs.
- **Target:** the target is **categorical** with 3 classes, so this is a **classification** problem, not regression.
- **Real-world problem:** naming the tumour type from one RNA measurement instead of several molecular lab tests.
- **Shape of the data:** there are far more genes (about 20,000) than patients (654). This "p >> n" shape makes overfitting the main risk throughout the project.

## 2. Exploratory Data Analysis (EDA)
EDA uses all patients, but **only to describe the data**. Nothing learned here is fed into the models.

### 2.1 Structure and statistical summary

In [ ]:
X.iloc[:5, :6]          # first 5 patients, first 6 genes

In [ ]:
gene_summary = X.describe().T              # one row per gene: count, mean, std, min, quartiles, max
gene_summary.describe().round(1)           # how those per-gene statistics vary across all genes

### 2.2 Missing values and duplicate records

In [ ]:
print("Missing values:", int(X.isna().sum().sum()))
print("Duplicate patient IDs:", int(X.index.duplicated().sum()))
print("Duplicate rows (identical profiles):", int(X.duplicated().sum()))
print("Genes that are zero in every patient:", int((X.sum() == 0).sum()))
print("Genes that are zero in over 90% of patients:", int(((X == 0).mean() > 0.9).sum()))

### 2.3 Distribution of the target and of the inputs

In [ ]:
counts = y.value_counts()
print(counts)

ax = counts.plot.bar(color=BLUE, edgecolor="black", rot=0, figsize=(6, 3.5))
ax.set_title("Patients per class")
ax.set_ylabel("Patients")
plt.show()

In [ ]:
values = X.values.ravel()                                        # every expression value in one long list
raw_values = values[values < np.percentile(values, 99)]          # drop the top 1% so the plot is readable
log_values = np.log2(values + 1)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))

ax[0].hist(raw_values, bins=60, color=GREY, edgecolor="black", linewidth=0.3)
ax[0].set_title("Raw values (below 99th percentile)")

ax[1].hist(log_values, bins=60, color=BLUE, edgecolor="black", linewidth=0.3)
ax[1].set_title("After log2(x + 1)")

plt.tight_layout()
plt.show()

print("Raw maximum:", int(values.max()))
print("Raw median:", np.median(values))

In [ ]:
fraction_expressed = (X > 0).mean()          # for each gene: share of patients where it is active

plt.figure(figsize=(6, 3.5))
plt.hist(fraction_expressed, bins=50, color=BLUE, edgecolor="black", linewidth=0.3)
plt.axvline(0.2, color="black", linestyle="--", label="20% threshold")
plt.xlabel("Fraction of patients with expression > 0")
plt.ylabel("Number of genes")
plt.title("How many patients express each gene")
plt.legend(frameon=False)
plt.show()

print("Genes expressed in at least 20% of patients:", int((fraction_expressed >= 0.2).sum()), "of", len(fraction_expressed))

### 2.4 Potential outliers
Raw values are extremely skewed, so outliers are checked on the **log scale** with the IQR rule (a value is flagged if it lies more than 1.5 x IQR outside the middle 50% of that gene).

In [ ]:
X_log_all = np.log2(X + 1)                   # log scale, used only to describe the data

q1 = X_log_all.quantile(0.25)
q3 = X_log_all.quantile(0.75)
iqr = q3 - q1
is_outlier = (X_log_all < q1 - 1.5 * iqr) | (X_log_all > q3 + 1.5 * iqr)

share_per_patient = is_outlier.mean(axis=1)
print(f"Share of all values flagged:            {is_outlier.values.mean():.1%}")
print(f"Patients with over 10% of genes flagged: {int((share_per_patient > 0.10).sum())}")

In [ ]:
marker_genes = ["CHI3L1", "RBP1", "EMP3"]            # well-known glioma marker genes
short_names = ["IDHwt", "non-codel", "codel"]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, gene in zip(axes, marker_genes):
    groups = [X_log_all.loc[y == name, gene] for name in classes]
    ax.boxplot(groups, patch_artist=True, boxprops={"facecolor": BLUE, "alpha": 0.6})
    ax.set_xticks([1, 2, 3], short_names)
    ax.set_title(gene)
    ax.set_ylabel("log2 expression")
plt.tight_layout()
plt.show()

### 2.5 Relationships and correlations
Three views:
- **PCA:** how the classes sit in the data overall.
- **Gene vs class:** which genes differ most between the classes.
- **Gene vs gene:** how strongly the top genes move together.

In [ ]:
X_scaled_all = StandardScaler().fit_transform(X_log_all)
pcs = PCA(n_components=2, random_state=SEED).fit_transform(X_scaled_all)

plt.figure(figsize=(6, 4.5))
for name in classes:
    rows = (y == name).values
    plt.scatter(pcs[rows, 0], pcs[rows, 1], s=12, alpha=0.7, color=CLASS_COLOURS[name], label=name)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA of tumours, coloured by class")
plt.legend(frameon=False)
plt.show()

print("Patients per class and tumour grade:")
print(pd.crosstab(y, grade))

In [ ]:
# ANOVA F-score of every gene against the class (descriptive only, never used to pick model features)
f_scores, _ = f_classif(X_log_all, y)
f_scores = pd.Series(f_scores, index=X_log_all.columns).fillna(0)
top_genes = f_scores.nlargest(30).index

# Patients sorted by class, genes scaled to z-scores so they share one colour range
class_number = y.map({name: i for i, name in enumerate(classes)})
patient_order = class_number.sort_values(kind="stable").index
heat = X_log_all.loc[patient_order, top_genes]
heat = (heat - heat.mean()) / heat.std()

class_sizes = y.value_counts().reindex(classes).values
class_ends = class_sizes.cumsum()

plt.figure(figsize=(10, 7))
plt.imshow(heat.T.values, aspect="auto", cmap="coolwarm", vmin=-3, vmax=3)
plt.yticks(range(30), top_genes, fontsize=7)
plt.xticks(class_ends - class_sizes / 2, classes)
for end in class_ends[:-1]:
    plt.axvline(end - 0.5, color="black", linewidth=1.5)
plt.colorbar(label="z-score of log2 expression")
plt.title("Top 30 class-discriminating genes")
plt.show()

In [ ]:
gene_corr = X_log_all[top_genes].corr()          # correlation between the 30 top genes

plt.figure(figsize=(7, 6))
plt.imshow(gene_corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.xticks(range(30), top_genes, rotation=90, fontsize=6)
plt.yticks(range(30), top_genes, fontsize=6)
plt.colorbar(label="Pearson correlation")
plt.title("Correlation between the top 30 genes")
plt.show()

pairs = gene_corr.where(np.triu(np.ones(gene_corr.shape, dtype=bool), k=1)).stack()
print(f"Gene pairs with |r| > 0.8: {int((pairs.abs() > 0.8).sum())} of {len(pairs)}")

### 2.6 Key observations from the EDA
- **Classes:** 256 / 229 / 169 patients, moderately uneven. So we use stratified splits and macro-F1.
- **Size:** about 20,000 genes for 654 patients (p >> n). Overfitting is the main risk.
- **Quality:** there are no missing values or duplicate patients. Many genes are almost always zero, and a 20% expression filter removes most of them.
- **Skew:** raw values reach almost 900,000 with a median of about 500, so a log transform is needed.
- **Outliers:** on the log scale, flagged values are spread across many genes and patients. They are natural biological variation, not data errors.
- **Structure:** IDHwt separates clearly in the PCA. The two IDH-mutant classes overlap and differ in fewer genes.
- **Correlation:** the top genes are strongly correlated with each other, so many genes carry overlapping information.
- **Grade:** IDHwt is mostly grade 4, so the classes partly track tumour severity.

## 3. Data Preprocessing
| Issue | Decision | Why |
|---|---|---|
| Missing values | Nothing to impute | None were found (2.2) |
| Duplicates | Nothing removed | No duplicate patients or profiles (2.2) |
| Skew and outliers | **log2(x + 1)**, no rows removed | Compresses huge values; extreme values are real biology, not errors |
| Categorical target | **Label encoding** (0 / 1 / 2) | Models (especially XGBoost) need numeric labels; inputs are already numeric |
| Different gene scales | **Standardisation** inside the LR and KNN pipelines | Needed for a penalised linear model and for distances; trees do not need it |

log2 and label encoding are fixed rules that learn nothing from the data, so they can be applied before the split. Standardisation **does** learn (mean and spread), so it lives inside each model's pipeline and is fitted on training data only (Step 7).

### 3.1 Log transform

In [ ]:
X_log = np.log2(X + 1)            # +1 so that zero stays zero

print("Largest value before:", round(X.values.max(), 1))
print("Largest value after: ", round(X_log.values.max(), 2))

### 3.2 Encode the target

In [ ]:
code_of = {name: i for i, name in enumerate(classes)}        # IDHwt = 0, IDHmut-non-codel = 1, IDHmut-codel = 2
print(code_of)

# The mapping is applied right after the split (Step 5), so the split itself uses the class names
y.map(code_of).value_counts().sort_index()

## 4. Feature Selection
**Which features we keep and why**
- **Remove rarely-active genes.** We drop genes that are active (> 0) in fewer than 20% of patients. Mostly-zero genes carry almost no information, add noise and make overfitting easier.
- **Keep the filter unsupervised.** It never looks at the labels.
- **Learn it from training data only.** It is the first step of every model pipeline, so in cross-validation each fold learns its own gene list.
- **No supervised selection.** The ANOVA ranking (2.5) is used only to describe the data. Choosing genes by their link to the target before cross-validation would leak label information.
- **Redundant genes are kept.** Correlated genes (2.5) are not dropped by hand. Each model handles them instead: L2 shrinkage (LR), PCA (KNN), and random gene subsets (Random Forest, XGBoost).

scikit-learn can only re-fit a step inside each cross-validation fold if the step is a small class with `fit` (learn which genes to keep) and `transform` (keep them). This is the **only class in the notebook**.

### 4.1 The low-expression filter

In [ ]:
class LowExpressionFilter(BaseEstimator, TransformerMixin):
    """Keep genes that are active (> 0) in at least `min_fraction` of the patients seen in fit()."""

    def __init__(self, min_fraction=0.2):
        self.min_fraction = min_fraction

    def fit(self, X, y=None):
        share_active = (X > 0).mean()                       # per gene, share of patients with expression > 0
        self.keep_ = (share_active >= self.min_fraction).values
        self.genes_ = X.columns[self.keep_]                 # names of the genes that are kept
        return self

    def transform(self, X):
        return X.loc[:, self.keep_]

### 4.2 Genes most related to the target (descriptive)

In [ ]:
f_scores.nlargest(10).round(1).rename("ANOVA F-score")

**How the features drive predictions.** The strongest genes split IDHwt from the IDH-mutant tumours, which matches the large gene switch-off caused by the IDH mutation. Separating codel from non-codel needs many weaker signals, such as lower activity of genes on chromosome arms 1p and 19q. That is why models that combine many genes are expected to do best.

## 5. Dataset Splitting
- **Ratio:** 70% train / 10% validation / 20% test. Stratified so each set keeps the class mix, with a fixed seed.
- **Training set:** fits the models and tunes them (5-fold cross-validation happens inside it).
- **Validation set:** used once, to choose between the five tuned models.
- **Test set:** used once at the end, for the final unbiased scores.
- **No leakage:** every learned step (gene filter, scaler, PCA, model) sits inside a pipeline fitted on training data only. The test set never influences any choice.

In [ ]:
# Step 1: put 20% of patients aside as the test set (stratified on the class names)
X_rest, X_te, y_rest, y_te = train_test_split(X_log, y, test_size=0.20, stratify=y, random_state=SEED)

# Step 2: from the remaining 80%, take 12.5% (= 10% of all patients) as the validation set
X_tr, X_va, y_tr, y_va = train_test_split(X_rest, y_rest, test_size=0.125, stratify=y_rest, random_state=SEED)

# Step 3: turn class names into the codes 0 / 1 / 2 (mapping from 3.2)
y_tr = y_tr.map(code_of)
y_va = y_va.map(code_of)
y_te = y_te.map(code_of)

print(f"Train:      {len(X_tr)} patients ({len(X_tr) / len(X):.0%})")
print(f"Validation: {len(X_va)} patients ({len(X_va) / len(X):.0%})")
print(f"Test:       {len(X_te)} patients ({len(X_te) / len(X):.0%})")

In [ ]:
class_mix = pd.DataFrame({
    "train": y_tr.value_counts(),
    "validation": y_va.value_counts(),
    "test": y_te.value_counts(),
}).sort_index()
class_mix.index = classes
class_mix

In [ ]:
# For information only: how many genes the filter keeps when it sees all training patients
genes_kept = LowExpressionFilter(0.2).fit(X_tr).genes_
print("Genes before filter:", X_tr.shape[1])
print("Genes kept on the full training set:", len(genes_kept))

## 6. Selection of Machine Learning Algorithms
Five supervised classifiers: a linear baseline, two other single models and two ensembles (one bagging, one boosting).

| Algorithm | How it works | Why it suits this data | Assumptions / limitations |
|---|---|---|---|
| **Logistic Regression (L2)** | Weighted sum of genes turned into class probabilities (softmax); the L2 penalty shrinks weights | Strong, simple baseline when features far outnumber samples | Assumes roughly linear class boundaries; needs scaled inputs |
| **Decision Tree** | A chain of yes/no questions on single genes | Easy to read; the reference point for the ensembles | High variance; overfits unless depth and leaf size are limited |
| **K-Nearest Neighbours** | Majority vote of the most similar training patients (in PCA space) | Simple, no training; tests whether similar profiles share a type | Distances weaken in high dimensions, hence PCA; sensitive to scaling |
| **Random Forest** (bagging) | Many trees on bootstrap samples and random gene subsets, votes averaged | Cuts a single tree's variance; copes with correlated genes | Less interpretable; slower; importances favour high-variance genes |
| **XGBoost** (boosting) | Shallow trees added one by one, each fixing the errors so far | Often the strongest on tabular data; cuts bias | Many hyperparameters; can overfit without a small learning rate |

## 7. Model Implementation and Training
Every model is a scikit-learn **Pipeline**: filter, then scaling/PCA where needed, then the classifier. The same steps are applied to train, validation and test data, and every step is fitted on training data only. Each model is stored with the hyperparameter values to try.

### 7.1 Logistic Regression

In [ ]:
models = {}      # model name -> (pipeline, hyperparameter values to try)

models["Logistic Regression"] = (
    Pipeline([
        ("filter", LowExpressionFilter(0.2)),
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(solver="lbfgs", max_iter=500, random_state=SEED)),
    ]),
    {"clf__C": [0.0001, 0.001, 0.01, 0.1, 1]},             # smaller C = stronger L2 penalty
)

### 7.2 Decision Tree

In [ ]:
models["Decision Tree"] = (
    Pipeline([
        ("filter", LowExpressionFilter(0.2)),
        ("clf", DecisionTreeClassifier(random_state=SEED)),
    ]),
    {
        "clf__max_depth": [3, 5, 8, None],                  # how many questions deep
        "clf__min_samples_leaf": [1, 5, 15],                # smallest group allowed at the end of a branch
        "clf__ccp_alpha": [0, 0.005, 0.02],                 # pruning strength
    },
)

### 7.3 K-Nearest Neighbours

In [ ]:
models["KNN"] = (
    Pipeline([
        ("filter", LowExpressionFilter(0.2)),
        ("scale", StandardScaler()),
        ("pca", PCA(random_state=SEED)),
        ("clf", KNeighborsClassifier()),
    ]),
    {
        "pca__n_components": [10, 30],                      # how many summary directions
        "clf__n_neighbors": [3, 7, 15, 25],                 # how many neighbours vote
        "clf__weights": ["uniform", "distance"],            # equal votes or closer neighbours count more
    },
)

### 7.4 Random Forest (bagging)

In [ ]:
models["Random Forest"] = (
    Pipeline([
        ("filter", LowExpressionFilter(0.2)),
        ("clf", RandomForestClassifier(random_state=SEED, n_jobs=1)),
    ]),
    {
        "clf__n_estimators": [200, 500],                    # number of trees
        "clf__max_features": ["sqrt", 0.2],                 # genes each split may look at
        "clf__min_samples_leaf": [1, 3, 8],
    },
)

### 7.5 XGBoost (boosting)

In [ ]:
models["XGBoost"] = (
    Pipeline([
        ("filter", LowExpressionFilter(0.2)),
        ("clf", XGBClassifier(objective="multi:softprob", tree_method="hist", subsample=0.8,
                              eval_metric="mlogloss", n_jobs=1, random_state=SEED)),
    ]),
    {
        "clf__n_estimators": [100, 300],                    # number of boosting rounds
        "clf__max_depth": [2, 4],
        "clf__learning_rate": [0.05, 0.2],                  # size of each correction step
        "clf__colsample_bytree": [0.2, 0.5],                # share of genes per tree
        "clf__reg_lambda": [1, 10],                         # L2 penalty on leaf weights
    },
)

### 7.6 Training with hyperparameter tuning
Every combination of settings is tried with **5-fold stratified cross-validation on the training set**, scored by **macro-F1**, which weights the three classes equally. The best setting of each model is then refitted on all training patients.

*This is the slow cell (roughly 30-75 minutes). Two parallel workers keep memory use moderate.*

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

best = {}         # model name -> pipeline refitted on all training patients with its best settings
cv_score = {}     # model name -> its cross-validated macro-F1

for name, (pipeline, grid) in models.items():
    search = GridSearchCV(pipeline, grid, cv=cv, scoring="f1_macro", n_jobs=2)
    search.fit(X_tr, y_tr)

    best[name] = search.best_estimator_
    cv_score[name] = search.best_score_

    print(f"{name:20s} CV macro-F1 = {search.best_score_:.3f}")
    print("   best settings:", search.best_params_)

**Reading the tuning output.**
- **Logistic Regression:** a small `C` means the data needs a strong penalty.
- **Decision Tree:** a shallow depth means a single tree must be kept simple to avoid memorising the training set.
- **At the edge of the grid:** a best value at the end of a range suggests the true optimum may lie just outside it.

## 8. Model Evaluation
Each tuned model is scored first on the **validation set**, which picks the final model. Then each is scored once on the **test set** for the final, unbiased scores.

### 8.1 Validation set: choose the final model

In [ ]:
rows = []
for name, model in best.items():
    val_pred = model.predict(X_va)
    rows.append({
        "Model": name,
        "CV macro-F1 (train)": cv_score[name],
        "Val accuracy": accuracy_score(y_va, val_pred),
        "Val macro-F1": f1_score(y_va, val_pred, average="macro"),
    })

val_table = pd.DataFrame(rows).set_index("Model").round(3)

selected = val_table["Val macro-F1"].idxmax()          # ties go to the model listed first
print("Selected model (best validation macro-F1):", selected)
val_table.sort_values("Val macro-F1", ascending=False)

### 8.2 Test set: predictions of every model

In [ ]:
preds = {}       # model name -> predicted class for each test patient (0 / 1 / 2)
probas = {}      # model name -> predicted probability of each class

for name, model in best.items():
    preds[name] = model.predict(X_te)
    probas[name] = model.predict_proba(X_te)

print("Test patients:", len(y_te))

### 8.3 Classification metrics
- **Accuracy, precision, recall, F1:** macro-averaged, so each class counts equally.
- **ROC-AUC:** how well the predicted probabilities rank the classes (one-vs-rest).
- **Robust single scores:** balanced accuracy, MCC and Cohen's kappa stay fair when classes are uneven.
- **Overfit gap:** training accuracy minus test accuracy.

In [ ]:
y_te_onehot = pd.get_dummies(y_te).values          # true classes as 0/1 columns, needed for PR-AUC

rows = []
for name, model in best.items():
    p, pr = preds[name], probas[name]
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_te, p),
        "Balanced acc": balanced_accuracy_score(y_te, p),
        "Macro precision": precision_score(y_te, p, average="macro"),
        "Macro recall": recall_score(y_te, p, average="macro"),
        "Macro-F1": f1_score(y_te, p, average="macro"),
        "ROC-AUC (ovr)": roc_auc_score(y_te, pr, multi_class="ovr", average="macro"),
        "PR-AUC": average_precision_score(y_te_onehot, pr, average="macro"),
        "MCC": matthews_corrcoef(y_te, p),
        "Cohen kappa": cohen_kappa_score(y_te, p),
        "Log-loss": log_loss(y_te, pr),
        "Train acc": accuracy_score(y_tr, model.predict(X_tr)),
    })

eval_matrix = pd.DataFrame(rows).set_index("Model")
eval_matrix["Overfit gap"] = eval_matrix["Train acc"] - eval_matrix["Accuracy"]
eval_matrix = eval_matrix.round(3)
eval_matrix.sort_values("Macro-F1", ascending=False)

### 8.4 Confusion matrices

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

for ax, name in zip(axes, preds):
    ConfusionMatrixDisplay.from_predictions(y_te, preds[name], ax=ax, labels=[0, 1, 2],
                                            display_labels=classes, cmap="Blues",
                                            colorbar=False, xticks_rotation=20)
    ax.set_title(name)

axes[-1].axis("off")          # 5 models, 6 panels: hide the empty one
plt.tight_layout()
plt.show()

### 8.5 Per-class precision, recall and F1 of the selected model

In [ ]:
print("Selected model:", selected)
print()
print(classification_report(y_te, preds[selected], target_names=classes))

### 8.6 How certain are the scores? (bootstrap 95% confidence intervals)
The 131 test patients are resampled 1,000 times with replacement, and macro-F1 is recomputed each time. Heavily overlapping intervals mean the data cannot tell two models apart.

In [ ]:
rng = np.random.default_rng(SEED)
y_true = y_te.values
n_patients = len(y_true)
ci = {}               # model name -> (lower, upper) of the 95% interval

for name in best:
    p = preds[name]
    scores = []
    for _ in range(1000):
        pick = rng.integers(0, n_patients, n_patients)         # resample test patients with replacement
        scores.append(f1_score(y_true[pick], p[pick], average="macro"))
    ci[name] = (np.percentile(scores, 2.5), np.percentile(scores, 97.5))

    print(f"{name:20s} macro-F1 = {eval_matrix.loc[name, 'Macro-F1']:.3f}"
          f"   95% CI [{ci[name][0]:.3f}, {ci[name][1]:.3f}]")

### 8.7 Interpretation: strengths and weaknesses
- **Errors:** most mistakes, if any, fall between the two IDH-mutant classes. This is expected from their overlap in the PCA.
- **Overfitting:** a large overfit gap means the model memorised training patients. KNN with distance weights always scores 100% on training data, so look at its test score.
- **Underfitting:** a low training accuracy together with a low cross-validation score, typical of a very shallow tree, means the model is too simple.
- **Small test set:** with 131 test patients, one mistake is worth about 0.8 accuracy points. Use the confidence intervals before calling one model better.

## 9. Model Comparison

### 9.1 Comparative summary

In [ ]:
ci_low = {name: ci[name][0] for name in ci}
ci_high = {name: ci[name][1] for name in ci}

comparison = pd.DataFrame({
    "CV macro-F1": pd.Series(cv_score),
    "Val macro-F1": val_table["Val macro-F1"],
    "Test macro-F1": eval_matrix["Macro-F1"],
    "CI low": pd.Series(ci_low),
    "CI high": pd.Series(ci_high),
    "Test ROC-AUC": eval_matrix["ROC-AUC (ovr)"],
    "Overfit gap": eval_matrix["Overfit gap"],
}).round(3)

comparison.sort_values("Test macro-F1", ascending=False)

In [ ]:
order = comparison.sort_values("Test macro-F1", ascending=False).index
x = np.arange(len(order))
width = 0.38

plt.figure(figsize=(8, 3.8))
plt.bar(x - width / 2, comparison.loc[order, "CV macro-F1"], width, color=GREY, edgecolor="black", label="CV (train)")
plt.bar(x + width / 2, comparison.loc[order, "Test macro-F1"], width, color=BLUE, edgecolor="black", label="Test")
plt.xticks(x, order)
plt.ylim(0.85, 1.0)
plt.ylabel("Macro-F1")
plt.title("Cross-validation vs test macro-F1")
plt.legend(frameon=False)
plt.show()

### 9.2 Trade-offs and the final choice
| Model | Performance | Complexity | Interpretability |
|---|---|---|---|
| Logistic Regression | High | Low | High: one weight per gene and class |
| Decision Tree | Lower | Very low | Very high: a few readable rules |
| KNN | Lowest | Low | Low: "similar patients" only |
| Random Forest | High | Medium | Medium: gene importances |
| XGBoost | High | High (many settings) | Medium: gene importances |

**Choice.** The final model is the one with the best **validation** macro-F1. That choice was fixed before the test set was opened, and it is not changed after seeing test scores, since doing so would leak the test set into the model choice. If the confidence intervals of the top models overlap, they perform about equally, and the simpler, more interpretable model is a sensible practical pick.

In [ ]:
m = eval_matrix.loc[selected]

# Most common mistake of the selected model: the largest off-diagonal cell of its confusion matrix
cm = confusion_matrix(y_te, preds[selected], labels=[0, 1, 2])
np.fill_diagonal(cm, 0)
true_i, pred_i = np.unravel_index(cm.argmax(), cm.shape)

print(f"Patients:              {len(X)} total -> {len(X_tr)} train / {len(X_va)} validation / {len(X_te)} test")
print(f"Selected model:        {selected}")
print(f"Test accuracy:         {m['Accuracy']:.3f}")
print(f"Test macro-F1:         {m['Macro-F1']:.3f}  (95% CI {ci[selected][0]:.3f} - {ci[selected][1]:.3f})")
print(f"Test ROC-AUC:          {m['ROC-AUC (ovr)']:.3f}")
print(f"Overfit gap:           {m['Overfit gap']:.3f}  (train accuracy - test accuracy)")
print()
print(f"Bagging vs single tree (RF vs DT):  {eval_matrix.loc['Random Forest', 'Macro-F1']:.3f} vs "
      f"{eval_matrix.loc['Decision Tree', 'Macro-F1']:.3f}")
print(f"Boosting vs bagging (XGB vs RF):    {eval_matrix.loc['XGBoost', 'Macro-F1']:.3f} vs "
      f"{eval_matrix.loc['Random Forest', 'Macro-F1']:.3f}")
if cm.max() > 0:
    print(f"Most common error:                  {classes[true_i]} predicted as {classes[pred_i]} "
          f"({cm.max()} patients)")

## 10. Feature Importance and Result Interpretation
Two complementary views:
- **Random Forest importance:** how much each gene helps the trees split the classes.
- **Logistic Regression coefficients:** which genes push a patient towards each class.

### 10.1 Random Forest: most important genes

In [ ]:
forest = best["Random Forest"].named_steps["clf"]
genes_used = best["Random Forest"].named_steps["filter"].genes_

importance = pd.Series(forest.feature_importances_, index=genes_used)
top10 = importance.nlargest(10)

plt.figure(figsize=(6, 4))
plt.barh(top10.index[::-1], top10.values[::-1], color=BLUE, edgecolor="black")
plt.xlabel("Importance (mean decrease in impurity)")
plt.title("Random Forest: top 10 genes")
plt.show()

top10.round(4)

### 10.2 Logistic Regression: genes that push towards each class

In [ ]:
lr = best["Logistic Regression"]
lr_genes = lr.named_steps["filter"].genes_
coefs = pd.DataFrame(lr.named_steps["clf"].coef_.T, index=lr_genes, columns=classes)

top_per_class = pd.DataFrame({name: coefs[name].nlargest(5).index for name in classes})
top_per_class.index = [f"#{i}" for i in range(1, 6)]
top_per_class

### 10.3 Looking at individual predictions

In [ ]:
proba_table = pd.DataFrame(probas[selected], index=X_te.index, columns=classes).round(3)
proba_table["True"] = [classes[i] for i in y_te]
proba_table["Predicted"] = [classes[i] for i in preds[selected]]

wrong = proba_table[proba_table["True"] != proba_table["Predicted"]]
print(f"Wrongly classified test patients ({selected}): {len(wrong)}")
display(wrong)

print("Five correctly classified examples:")
proba_table[proba_table["True"] == proba_table["Predicted"]].head()

### 10.4 Are the patterns meaningful?
- **RBP1:** the top gene is known to be switched off in almost all IDH-mutant gliomas, so it is a natural marker of IDH status.
- **Chromosome arms 1p and 19q:** most of the other top genes lie on these arms. Losing one copy of them lowers their activity. The importance pattern is therefore **consistent with** the biological effects of 1p/19q codeletion.
- **Uncertain errors:** wrong predictions usually come with lower top probabilities. These are the cases that would need a second look in practice.
- **Association, not causation:** importances and coefficients show which genes the models *use*. They do not prove that a gene *causes* the tumour type. Correlated genes also share credit between them.

## Conclusion
- **Main result:** tumour gene expression alone separates the three glioma types with high accuracy on patients never used for training, tuning or model choice.
- **Single tree vs ensembles:** ensembles clearly beat a single tree.
- **Linear baseline:** the penalised linear model is as strong as the ensembles, which is typical when genes far outnumber patients.
- **Biology:** the genes the models rely on are consistent with known tumour biology.

**Limitations**
- About 650 patients from one TCGA cohort and one split, with no external validation or batch correction.
- The validation set is small (66 patients), so the choice between close models is noisy.
- IDHwt is mostly grade 4, so part of what is learned may be tumour severity.
- The tuning grids are small, and some best values sit at the edge of their range.
- This is a research exercise, not a validated diagnostic tool.

**Next steps**
- Validate on an independent cohort.
- Widen the tuning ranges.
- Try repeated cross-validation.
- Add SVM models.